# Assignment 3: Fuzzy Logic Control of a Simulated Robotic Arm
Input: error = target angle − current angle. Output: angle change (move) per step.

In [ ]:
# !pip install scikit-fuzzy
import numpy as np
import skfuzzy as fuzz
from skfuzzy import control as ctrl
import matplotlib.pyplot as plt

In [ ]:
# 1. Variables
error = ctrl.Antecedent(np.arange(-180, 181, 1), 'error')
move  = ctrl.Consequent(np.arange(-20, 21, 1), 'move')

# 2. Membership functions
error['neg_big']   = fuzz.trimf(error.universe, [-180, -180, -45])
error['neg_small'] = fuzz.trimf(error.universe, [-90, -30, 0])
error['zero']      = fuzz.trimf(error.universe, [-10, 0, 10])
error['pos_small'] = fuzz.trimf(error.universe, [0, 30, 90])
error['pos_big']   = fuzz.trimf(error.universe, [45, 180, 180])

move['neg_big']   = fuzz.trimf(move.universe, [-20, -20, -8])
move['neg_small'] = fuzz.trimf(move.universe, [-12, -5, 0])
move['zero']      = fuzz.trimf(move.universe, [-2, 0, 2])
move['pos_small'] = fuzz.trimf(move.universe, [0, 5, 12])
move['pos_big']   = fuzz.trimf(move.universe, [8, 20, 20])

# 3. Rules: far from target -> move fast, close -> move slowly
rules = [
    ctrl.Rule(error['neg_big'],   move['neg_big']),
    ctrl.Rule(error['neg_small'], move['neg_small']),
    ctrl.Rule(error['zero'],      move['zero']),
    ctrl.Rule(error['pos_small'], move['pos_small']),
    ctrl.Rule(error['pos_big'],   move['pos_big']),
]
arm = ctrl.ControlSystemSimulation(ctrl.ControlSystem(rules))

def simulate(start, target, max_steps=100):
    angle = start
    history = [angle]
    for step in range(1, max_steps + 1):
        e = target - angle
        arm.input['error'] = e
        arm.compute()
        angle += arm.output['move']
        history.append(angle)
        print(f"Step {step:3d} | Angle = {angle:7.2f} | Error = {target - angle:7.2f}")
        if abs(target - angle) < 0.5:
            print(f"Target {target} reached in {step} steps")
            break
    return history

In [ ]:
current_angle, target_angle, history = 0.0, 120.0, None

while True:
    print("\n========= ROBOTIC ARM MENU =========")
    print(f"Current angle = {current_angle:.2f}, Target = {target_angle:.2f}")
    print("1. Set current and target angle")
    print("2. Run fuzzy controller simulation")
    print("3. Plot arm angle vs steps")
    print("4. Show membership functions")
    print("0. Exit")
    ch = input("Enter your choice: ").strip()
    try:
        if ch == '1':
            current_angle = float(input("Enter current angle (0-180): "))
            target_angle  = float(input("Enter target angle (0-180): "))
            if not (0 <= current_angle <= 180 and 0 <= target_angle <= 180):
                print("Angles must be between 0 and 180. Resetting.")
                current_angle, target_angle = 0.0, 120.0
        elif ch == '2':
            history = simulate(current_angle, target_angle)
        elif ch == '3':
            if history is None:
                print("Run the simulation first (option 2).")
                continue
            plt.plot(history, marker='o')
            plt.axhline(target_angle, color='r', linestyle='--', label='Target')
            plt.xlabel('Step'); plt.ylabel('Arm angle (deg)')
            plt.title('Fuzzy Controlled Robotic Arm'); plt.legend(); plt.show()
        elif ch == '4':
            error.view(); move.view(); plt.show()
        elif ch == '0':
            print("Exiting...")
            break
        else:
            print("Invalid choice")
    except ValueError:
        print("Please enter numbers only")